# Musanga V4 — NB04 V4.007: multispectral analysis using original plots and circles

**One active modelling workflow:** Sections **06A → 06B → 07A → 07B** start directly from the original exhaustive plot and circle polygons with frozen folds, and use bounded native-window reads (no large raster operations). All available MS surveys, including MOS 5–6, are considered; band-1 zeros remain invalid rather than being treated as reflectance. No full MS U-Net training.

Sections **00–05** are independent input inventory and quality-control checks; run section 00 first for setup, but you do not need to rerun the existing QA if it is already complete.

**Run: 00, 06A, 06B, 07A, 07B.**

**Topology QA:** invalid original crown polygons are repaired with Shapely `make_valid` and polygon-only extraction, with area-change audit and fail-closed >5% alteration. Original GIS files remain unchanged.


In [ ]:
# 00 — Reuse NB00 inventory and frozen ROI; metadata only, no image read
from pathlib import Path
import json, math, re, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
DATA=ROOT/'Data'
NB00=ROOT/'Outputs/v4/00_input_audit'
OUT=ROOT/'Outputs/v4/04_ms_qa_v4_007'
OUT.mkdir(parents=True,exist_ok=True)
REG=NB00/'00_ms_registry.json'
HEADERS=NB00/'00_ms_raster_headers.csv'
OVERLAP=NB00/'00_ms_rgb_extent_audit.csv'
assert REG.is_file() and HEADERS.is_file() and OVERLAP.is_file(),(
    'First run NB00 MS inventory cells; missing',REG,HEADERS,OVERLAP)
ms_manifest=json.loads(REG.read_text())
ms=pd.read_csv(HEADERS).fillna({'error':''})
overlap=pd.read_csv(OVERLAP)
ROI_PATH=DATA/'Ortho_RGB'/'ROI.shp'
roi=gpd.read_file(ROI_PATH)
assert roi.crs is not None and len(roi)>0
RGB_REFERENCE=ms_manifest['rgb_reference']
RGB_DIR=DATA/'Ortho_RGB'
RGB_FILES={group:RGB_DIR/name for group,name in RGB_REFERENCE.items()}
assert all(p.is_file() for p in RGB_FILES.values()),RGB_FILES
EDGE_DISTANCE_M=(1,3,5,10,20,30,50)
print('Available surveys:', sorted(ms.group.dropna().unique()))
print('NB00 MS records:',len(ms),'| paired extent records:',len(overlap))
display(ms[['group','band_hint','name','bands','dtype','nodata','error']].head(40))


In [ ]:
# 01 — Band headers only, ZERO source pixel reads
rows=[]
for rec in ms.itertuples():
    if getattr(rec,'error',''):continue
    path=Path(rec.path)
    if not path.is_file():continue
    with rasterio.Env(GDAL_CACHEMAX=32):
        with rasterio.open(path) as ds:
            for b in range(1,ds.count+1):
                rows.append(dict(group=rec.group,file=path.name,path=str(path),
                    band=b,band_hint=rec.band_hint,
                    description=ds.descriptions[b-1],
                    colorinterp=str(ds.colorinterp[b-1]),
                    dtype=ds.dtypes[b-1],scale=ds.scales[b-1],offset=ds.offsets[b-1],
                    nodata=ds.nodatavals[b-1],crs=str(ds.crs),
                    width=ds.width,height=ds.height,
                    pixel_x_m=abs(ds.transform.a),pixel_y_m=abs(ds.transform.e),
                    block_shape=str(ds.block_shapes[b-1]),
                    pixel_distribution='NOT SAMPLED',file_bytes=path.stat().st_size))
    print('HEADERS:',rec.group,rec.name,flush=True)
BAND_AUDIT=pd.DataFrame(rows)
assert not BAND_AUDIT.empty
BAND_AUDIT.to_csv(OUT/'04_ms_band_metadata_only.csv',index=False)
display(BAND_AUDIT[['group','file','band','band_hint','description','dtype',
                    'scale','offset','nodata','pixel_x_m','block_shape']])
print('No pixel data read, no band-value statistics inferred.')


In [ ]:
# 02 — ROI / RGB / MS BOUNDING-BOX overlap ONLY, ZERO pixel reads
from shapely.geometry import box
from shapely.ops import unary_union
from rasterio.warp import transform_bounds
summary=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        study=unary_union(roi.to_crs(rgb.crs).geometry).intersection(box(*rgb.bounds))
        area=float(study.area)
        if area<=0:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            with rasterio.Env(GDAL_CACHEMAX=32):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    bb=box(*transform_bounds(src.crs,rgb.crs,*src.bounds,densify_pts=21))
                    frac=float(study.intersection(bb).area/area)
                    summary.append(dict(group=group,file=rec.name,band_hint=rec.band_hint,
                        nominal_bands=int(src.count),roi_bbox_overlap_fraction=frac,
                        rgb_roi_area_ha=area/1e4,valid_pixel_coverage='NOT TESTED',
                        registration='NOT TESTED'))
            print('BBOX:',group,rec.name,round(frac,3),flush=True)
SUMMARY=pd.DataFrame(summary)
assert not SUMMARY.empty
SUMMARY.to_csv(OUT/'04_ms_rgb_roi_bbox_only.csv',index=False)
display(SUMMARY.sort_values(['group','file']).round(3))
print('Bounding-box overlap is only an upper limit on valid MS pixel coverage.')


In [ ]:
# 03 — Metadata coverage figure, NO pixel reads
d=SUMMARY.sort_values(['group','file'])
fig,ax=plt.subplots(figsize=(9,max(4,len(d)*.23)))
ax.barh(np.arange(len(d)),d.roi_bbox_overlap_fraction.to_numpy())
ax.set_yticks(np.arange(len(d)))
ax.set_yticklabels([f"{r.group}: {r.file[:40]}" for r in d.itertuples()],fontsize=7)
ax.invert_yaxis()
ax.set_xlim(0,1.02)
ax.set_xlabel('Bounding-box overlap with RGB ∩ ROI (not actual valid coverage)')
ax.set_title('MS inventory — metadata only')
fig.tight_layout()
fig.savefig(OUT/'04_ms_bbox_metadata_summary.png',dpi=120,bbox_inches='tight')
plt.show()
plt.close(fig)
print('DONE:',OUT)


In [ ]:
# 04 — STRICTLY BOUNDED native-window MS sampling; do not resample whole images.
# 25 files x at most 3 windows x <= 2/4 bands x 64x64 pixels.
# Full-width strip TIFFs may decode an entire scanline internally; never request
# hundreds of rows, use WarpedVRT, or keep open handles/arrays between files.
from rasterio.warp import transform as transform_coords
from rasterio.windows import Window
from shapely.geometry import Point,box
from shapely.ops import unary_union
import gc

WINDOW_SIZE=64
MAX_PATCHES_PER_FILE=3
QA_ROWS=[]
QA_PATCHES=[]
QA_WARNINGS=[]
for group,records in ms.groupby('group',sort=True):
    if group not in RGB_FILES:continue
    with rasterio.open(RGB_FILES[group]) as rgb:
        crs=rgb.crs
        study=unary_union(roi.to_crs(crs).geometry).intersection(box(*rgb.bounds))
        if study.is_empty:continue
        for rec in records.itertuples():
            if getattr(rec,'error',''):continue
            path=Path(rec.path)
            if not path.is_file():continue
            print('NATIVE WINDOWS:',group,path.name,flush=True)
            # Only one source open; cap GDAL's decompressed block cache at 16 MB.
            with rasterio.Env(GDAL_CACHEMAX=16):
                with rasterio.open(path) as src:
                    if src.crs is None:continue
                    from rasterio.warp import transform_bounds
                    source_box=box(*transform_bounds(src.crs,crs,*src.bounds,densify_pts=21))
                    relevant=study.intersection(source_box)
                    if relevant.is_empty:continue
                    bb=relevant.bounds
                    choices=[relevant.representative_point()]
                    # Deterministic geographic candidates, no raster samples/read.
                    for u,v in [(0.25,0.25),(0.75,0.75),(0.25,0.75),
                                (0.75,0.25),(0.5,0.5),(0.35,0.65),(0.65,0.35)]:
                        pt=Point(bb[0]+u*(bb[2]-bb[0]),bb[1]+v*(bb[3]-bb[1]))
                        if relevant.covers(pt):choices.append(pt)
                    seen=set()
                    count=0
                    for pt in choices:
                        if count>=MAX_PATCHES_PER_FILE:break
                        if src.crs==crs:x,y=pt.x,pt.y
                        else:
                            mx,my=transform_coords(crs,src.crs,[pt.x],[pt.y])
                            x,y=mx[0],my[0]
                        row,col=src.index(x,y)
                        if row<0 or col<0 or row>=src.height or col>=src.width:continue
                        col=int(np.clip(col-WINDOW_SIZE//2,0,max(0,src.width-WINDOW_SIZE)))
                        row=int(np.clip(row-WINDOW_SIZE//2,0,max(0,src.height-WINDOW_SIZE)))
                        key=(row//WINDOW_SIZE,col//WINDOW_SIZE)
                        if key in seen:continue
                        seen.add(key)
                        win=Window(col,row,min(WINDOW_SIZE,src.width-col),
                                   min(WINDOW_SIZE,src.height-row))
                        # Never read all bands at once; no large intermediate arrays.
                        pixel_arrays=[]
                        for band in range(1,src.count+1):
                            a=src.read(band,window=win,masked=True)
                            v=np.asarray(a.data)
                            valid=(~np.ma.getmaskarray(a))&np.isfinite(v)
                            vals=v[valid]
                            unique=np.unique(v) if v.size<=4096 else np.array([])
                            QA_ROWS.append(dict(group=group,file=path.name,
                                band_hint=rec.band_hint,band=band,patch=count,
                                row=row,col=col,n_window=int(v.size),
                                n_valid=int(valid.sum()),
                                valid_fraction=float(valid.mean()),
                                min=float(vals.min()) if len(vals) else np.nan,
                                p05=float(np.percentile(vals,5)) if len(vals) else np.nan,
                                median=float(np.median(vals)) if len(vals) else np.nan,
                                p95=float(np.percentile(vals,95)) if len(vals) else np.nan,
                                max=float(vals.max()) if len(vals) else np.nan,
                                zero_frac_valid=float(np.mean(vals==0)) if len(vals) else np.nan,
                                fraction_65535=float(np.mean(vals==65535)) if len(vals) else np.nan,
                                n_unique=int(len(unique)),
                                unique_values=str(unique[:12].tolist()) if len(unique)<=12 else 'many'))
                            # Caution: 2-band TIFF may be intensity + alpha. Report, do
                            # not pre-assign alpha without a diagnostic.
                            del a,v,valid,vals
                        count+=1
                    if count==0:QA_WARNINGS.append((group,path.name,'No in-footprint native patch'))
            print('  COMPLETED:',count,'native 64x64 patches; GDAL cache capped',flush=True)
            gc.collect()

PATCH_QA=pd.DataFrame(QA_ROWS)
assert not PATCH_QA.empty,'No valid sample windows in any MS raster'
PATCH_QA.to_csv(OUT/'04_native_patch_band_statistics.csv',index=False)
summ=(PATCH_QA.groupby(['group','file','band_hint','band'],dropna=False)
      .agg(n_patches=('patch','nunique'),mean_valid=('valid_fraction','mean'),
           median_p05=('p05','median'),median=('median','median'),
           median_p95=('p95','median'),mean_zero=('zero_frac_valid','mean'),
           mean_65535=('fraction_65535','mean'),
           median_n_unique=('n_unique','median')).reset_index())
summ.to_csv(OUT/'04_native_patch_summary.csv',index=False)
display(summ.round(3))
if QA_WARNINGS:print('NO VALID WINDOWS:',QA_WARNINGS)
print('Sample distributions are NOT proof of calibrated reflectance or registration.')
print('Native-window audit complete:',OUT)


In [ ]:
# 05 — Alpha-aware paired spectral QA (MOS 5–6 red and MOS 1–2 red-edge duplicates)
# Read only 16x16 source windows, at most 36 coordinates per survey.
# A zero in band 1 with alpha == 0 is TRANSPARENT, not evidence of corruption.
from shapely.geometry import Point
from rasterio.warp import transform as warp_coords
from rasterio.windows import Window
from itertools import combinations
import gc
SAMPLE_N=6
SIZE=16
qa_rows=[]
groups_to_check=('MOS_5_6_MS','MOS_1_2_MS')
for group in groups_to_check:
    recs=ms[(ms.group==group) & (ms.band_hint.isin(['red','green','nir','red_edge']))].copy()
    assert not recs.empty
    rgb_path=RGB_FILES[group]
    with rasterio.open(rgb_path) as rgb:
        base_crs=rgb.crs
        study=unary_union(roi.to_crs(base_crs).geometry).intersection(box(*rgb.bounds))
        # Limit to MS union intersection: prevents wasting all samples outside imagery.
        boxes=[]
        for rec in recs.itertuples():
            with rasterio.open(rec.path) as src:
                boxes.append(box(*transform_bounds(src.crs,base_crs,*src.bounds,densify_pts=21)))
        footprint=study.intersection(unary_union(boxes))
        assert not footprint.is_empty,group
        xmin,ymin,xmax,ymax=footprint.bounds
        test_points=[]
        for fy in np.linspace(.08,.92,SAMPLE_N):
            for fx in np.linspace(.08,.92,SAMPLE_N):
                pt=Point(xmin+fx*(xmax-xmin),ymin+fy*(ymax-ymin))
                if footprint.covers(pt):test_points.append(pt)
        if not test_points:
            test_points=[footprint.representative_point()]
    print('ALPHA QA:',group,'candidate shared locations',len(test_points),flush=True)
    # Open only a single raster at a time. Save tiny arrays for cross-band QA.
    sampled={}
    for rec in recs.itertuples():
        collected={}
        with rasterio.Env(GDAL_CACHEMAX=8):
            with rasterio.open(rec.path) as src:
                for j,pt in enumerate(test_points):
                    if src.crs==base_crs: xx,yy=pt.x,pt.y
                    else:
                        x2,y2=warp_coords(base_crs,src.crs,[pt.x],[pt.y])
                        xx,yy=x2[0],y2[0]
                    row,col=src.index(xx,yy)
                    if row<SIZE or col<SIZE or row>=src.height-SIZE or col>=src.width-SIZE:continue
                    win=Window(col-SIZE//2,row-SIZE//2,SIZE,SIZE)
                    # Read band 1 and (when present) alpha-like band 2 only.
                    v=src.read(1,window=win,masked=True)
                    z=np.asarray(v.data)
                    valid=(~np.ma.getmaskarray(v)) & np.isfinite(z)
                    alpha=None
                    if src.count>=2:
                        a=src.read(2,window=win,masked=False)
                        alpha=np.asarray(a)
                        # Explicitly treat alpha=0 as invisible. Non-zero may include
                        # partial transparency; record it separately.
                        valid &= alpha>0
                    values=z[valid]
                    qa_rows.append(dict(group=group,file=rec.name,band=rec.band_hint,
                        location=j,valid_fraction=float(valid.mean()),
                        valid_pixels=int(valid.sum()),
                        alpha_zero_fraction=float(np.mean(alpha==0)) if alpha is not None else np.nan,
                        alpha_full_fraction=float(np.mean(alpha==255)) if alpha is not None else np.nan,
                        signal_median=float(np.median(values)) if len(values) else np.nan,
                        signal_zero_fraction=float(np.mean(values==0)) if len(values) else np.nan))
                    collected[j]=(z.copy(),valid.copy())
        sampled[rec.name]=collected
        print('  ',rec.name,'windows',len(collected),'valid locations',
              sum(mask.sum()>0 for z,mask in collected.values()),flush=True)
        gc.collect()
    if group=='MOS_1_2_MS':
        rededges=[p for p in recs[recs.band_hint.eq('red_edge')].name]
        if len(rededges)==2:
            a,b=(sampled[x] for x in rededges)
            shared=sorted(set(a)&set(b))
            different=0;valid_pixels=0
            for k in shared:
                za,ma=a[k];zb,mb=b[k]
                joint=ma&mb
                different+=int(np.count_nonzero(za[joint]!=zb[joint]))
                valid_pixels+=int(joint.sum())
            print('DUPLICATE RED EDGE: differing shared valid sampled pixels:',
                  different,'/',valid_pixels,'across',len(shared),'windows')
            if valid_pixels==0:
                print('  INCONCLUSIVE: no shared valid pixels sampled.')
            elif different==0:
                print('  Sampled values identical; full-file identity NOT proven.')
    del sampled
qa=pd.DataFrame(qa_rows)
qa.to_csv(OUT/'04_alpha_aware_paired_QA.csv',index=False)
aggregate=(qa.groupby(['group','band','file'],as_index=False).agg(
    n_windows=('location','nunique'),
    windows_with_valid=('valid_pixels',lambda v:int((v>0).sum())),
    mean_valid_fraction=('valid_fraction','mean'),
    mean_alpha_zero_fraction=('alpha_zero_fraction','mean'),
    median_signal=('signal_median','median')))
display(aggregate.round(3))
print('QA saved:',OUT/'04_alpha_aware_paired_QA.csv')
print('NOTE: these are sparse native windows, not an estimate of full-area valid coverage.')


## Interpretation after cell 05

The previous cell 04 can report mean_valid=1 even when both spectral values and alpha are zero. Cell 05 explicitly checks the second channel as **candidate alpha**: pixels with zero in channel 2 are rejected. A high fraction of zero-alpha windows for MOS 5–6 red would indicate missing coverage, **not necessarily radiometric corruption**.

When valid red pixels exist, compare their paired local signals with green, NIR and red-edge, and then inspect RGB/MS alignment before computing spectral indices. MOS 1–2 duplicate red-edge filenames are tested at identical native locations; matching samples are not proof of full-file identity. No original dataset is modified.


## 06 — Full annotation-unit audit and matched RGB–MS sampling

**Run cells 00–05 for optional QA, then 06A, 06B, 07A and 07B for the model experiment.**

This is a new experiment rather than extending the underpowered cached-chip test. It uses original plot and circle geometries and their frozen outer folds. All eligible units are enumerated; each contributes a limited number of labelled positive and negative **representative crown/interior positions**, keeping per-unit influence bounded. Circles overlapping field plots are excluded to avoid shared labels. Training and validation units are separated by fold; training units within 15 m of held-out annotation units are removed. No partial labels.

**MOS 5–6:** band 1 is tested on small native windows as requested, *without requiring band 2 alpha*. A zero red-band window is considered unusable; no NDVI with red=0. Results include both 3-band (green/red-edge/NIR) and four-band comparisons where actual red signal exists. All other surveys also use band 1, with band 2 considered an optional transparency diagnostic. RGB–MS alignment and DN radiometry remain unverified.

**This is a first spatial unit-based screening**, not final canopy-cover validation or U-Net comparison. Multiple sampled points inside the same unit are not independent observations.


In [ ]:
# 06A — All frozen exhaustive annotation units and per-acquisition coverage
from shapely.geometry import box, Point
from shapely.ops import unary_union
from shapely.strtree import STRtree
from rasterio.warp import transform_bounds
from rasterio.windows import Window
from rasterio.transform import rowcol
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import hashlib, gc, os
from collections import defaultdict
REF=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT=json.loads((REF/'01_field78_reference_contract.json').read_text())
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
PF=pd.read_csv(CONTRACT['folds_csv'])
PLOTS=PLOTS.merge(PF[['source_id','site_key','cv_fold']],left_on='Plot_name',
                  right_on='source_id',validate='one_to_one')
CIRCLES=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
CF=pd.read_csv(REF/'01_field78_circle_folds.csv')
CIRCLES['circle_id']=np.arange(len(CIRCLES))
CIRCLES=CIRCLES.merge(CF[['circle_id','site_key','cv_fold',
                           'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs('EPSG:32635')
CROWN_CIRC=gpd.read_file(NB00['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(PLOTS)==78 and len(CIRCLES)==34
assert {'Class_ID'}.issubset(CROWNS.columns)
# Topology-safe polygon repair. Some digitized crowns have self-intersections.
# Only polygonal components are kept; never buffer(0) blindly or silently
# substitute an arbitrary boundary. Log repaired feature counts and area change.
from shapely import make_valid
from shapely.errors import GEOSException
def polygon_only(g):
    if g is None or g.is_empty:return None
    if g.geom_type in ('Polygon','MultiPolygon'):return g
    polygons=[]
    for z in getattr(g,'geoms',[]):
        p=polygon_only(z)
        if p is not None:polygons.append(p)
    return unary_union(polygons) if polygons else None
REPAIR_REPORT=[]
def safe_geom(g,label):
    if g is None or g.is_empty:return None
    old_area=float(g.area)
    if not g.is_valid:
        repaired=polygon_only(make_valid(g))
        if repaired is None or repaired.is_empty:
            raise RuntimeError(f'Cannot repair polygon {label}')
        change=abs(repaired.area-old_area)/max(old_area,1e-9)
        REPAIR_REPORT.append(dict(feature=label,area_before=old_area,
                                  area_after=float(repaired.area),
                                  area_rel_change=change))
        if change>.05:
            raise RuntimeError(f'Polygon repair changes area by {change:.1%}: {label}')
        g=repaired
    return g
for frame,name in [(PLOTS,'plot'),(CIRCLES,'circle'),
                   (CROWNS,'plot_crown'),(CROWN_CIRC,'circle_crown')]:
    frame.geometry=[safe_geom(g,f'{name}:{i}') for i,g in enumerate(frame.geometry)]
    if frame.geometry.isna().any():raise RuntimeError(f'Null geometry after repair: {name}')
pd.DataFrame(REPAIR_REPORT,columns=['feature','area_before','area_after',
   'area_rel_change']).to_csv(OUT/'06_polygon_repair_audit.csv',index=False)
print('Repaired',len(REPAIR_REPORT),'invalid polygon geometries; maximum area change',
      max([x['area_rel_change'] for x in REPAIR_REPORT],default=0))

# Use only geometry interior of existing exhaustive plot/circle; no partial crowns.
CROWNS=CROWNS.loc[CROWNS.geometry.notna()&~CROWNS.geometry.is_empty].copy()
CROWN_CIRC=CROWN_CIRC.loc[CROWN_CIRC.geometry.notna()&~CROWN_CIRC.geometry.is_empty].copy()
cgeoms=list(CROWNS.geometry)
ctree=STRtree(cgeoms)
ccgeoms=list(CROWN_CIRC.geometry)
cctree=STRtree(ccgeoms)
plot_geoms=list(PLOTS.geometry)
plot_tree=STRtree(plot_geoms)
RNG=np.random.default_rng(20261117)
SAMPLE_PER_CLASS_PER_UNIT=3
GUARD_M=15.0
NEW_OUT=OUT/'06_full_reference_unit_screen'
NEW_OUT.mkdir(parents=True,exist_ok=True)
GROUPS=['Baego_MS','MOS_1_2_MS','MOS_3_4_MS','MOS_5_6_MS',
        'Yangambi_YBJoseph_right_MS','Yangambi_YBForgeo_right_MS']
SITE_MAP={'Baego_MS':'Baego','MOS_1_2_MS':'MOS','MOS_3_4_MS':'MOS',
          'MOS_5_6_MS':'MOS','Yangambi_YBJoseph_right_MS':'Yangambi',
          'Yangambi_YBForgeo_right_MS':'Yangambi'}
MOSAIC_KEY={'Baego_MS':'Baego.tif',
            'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
            'MOS_3_4_MS':'Mos_3_4_transparent_mosaic_group1_georef.tif',
            'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
            'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
            'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
def band_registry(group):
    part=ms[ms.group.eq(group)]
    if group=='Baego_MS':
        assert len(part)==1
        with rasterio.open(part.iloc[0].path) as ds:
            assert tuple(str(x).lower().replace(' ','') for x in ds.descriptions)==(
                'red','green','rededge','nir')
        return {b:Path(part.iloc[0].path) for b in ('red','green','red_edge','nir')}
    result={}
    for b in ('red','green','red_edge','nir'):
        cand=part[part.band_hint.eq(b)]
        if len(cand)>1 and b=='red_edge' and group=='MOS_1_2_MS':
            cand=cand[cand.name.eq('mlt_Mos_1_2_transparent_mosaic_red edge_georef.tif')]
        assert len(cand)==1,(group,b,list(cand.name))
        result[b]=Path(cand.iloc[0].path)
    return result
BANDS={group:band_registry(group) for group in GROUPS}
def bbox_for_group(group):
    boxes=[]
    rgbpath=DATA/'Ortho_RGB'/MOSAIC_KEY[group]
    with rasterio.open(rgbpath) as rgb:
        rgb_box=box(*rgb.bounds); rgbcrs=rgb.crs
    for p in set(BANDS[group].values()):
        with rasterio.open(p) as ds:
            boxes.append(box(*transform_bounds(ds.crs,rgbcrs,*ds.bounds,densify_pts=21)))
    footprint=rgb_box.intersection(unary_union(boxes)) # coarse bbox only
    return footprint,rgbcrs
UNIT_ROWS=[]
for group in GROUPS:
    footprint,crs=bbox_for_group(group)
    assert crs.to_epsg()==32635,(group,crs)
    site=SITE_MAP[group]
    for i,p in PLOTS[PLOTS.site_key.eq(site)].iterrows():
        geo=p.geometry
        if geo is None or geo.is_empty or not geo.intersects(footprint):continue
        region=geo.intersection(footprint)
        if region.is_empty:continue
        UNIT_ROWS.append(dict(group=group,site=site,kind='plot',uid=str(p.Plot_name),
            fold=int(p.cv_fold),plot_index=int(i),circle_index=-1,
            region=region,full_geometry=geo))
    for i,c in CIRCLES[CIRCLES.site_key.eq(site)].iterrows():
        geo=c.geometry
        if geo is None or geo.is_empty or not geo.intersects(footprint):continue
        if bool(c.cross_fold_plot_intersection):continue
        # Exclude any other intersection with field plot supervision.
        if len(plot_tree.query(geo,predicate='intersects')):continue
        region=geo.intersection(footprint)
        if region.is_empty:continue
        UNIT_ROWS.append(dict(group=group,site=site,kind='circle',
            uid=str(c.circle_id),fold=int(c.cv_fold),plot_index=-1,
            circle_index=int(i),region=region,full_geometry=geo))
units=pd.DataFrame(UNIT_ROWS)
assert not units.empty
units['unit_key']=units.apply(lambda r:f'{r.kind}:{r.uid}',axis=1)
assert not units.duplicated(['group','unit_key']).any()
print('ALL ELIGIBLE ORIGINAL UNITS IN MS BOUNDING FOOTPRINT:')
display(units.groupby(['group','kind','fold']).size().unstack(fill_value=0))
summary=units.drop(columns=['region','full_geometry'])
summary.to_csv(NEW_OUT/'04_all_annotation_units_bbox_audit.csv',index=False)
print('Note: these are bbox intersections; final valid-data count follows sampling.')


In [ ]:
# 06B — Original annotation polygons: sample representative positive/negative positions
# Candidate points from every eligible plot or circle; avoid overlaying raw mosaics.
# Plot positives/negatives are the explicitly labelled Class_ID crown polygons.
# Circle negatives are remaining exhaustively labelled area excluding all positive crowns.
def point_candidates(region,n,seed):
    if region.is_empty:return []
    pts=[region.representative_point()]
    if len(pts)>=n:return pts
    xmin,ymin,xmax,ymax=region.bounds
    rng=np.random.default_rng(seed)
    for _ in range(120):
        if len(pts)>=n:break
        p=Point(rng.uniform(xmin,xmax),rng.uniform(ymin,ymax))
        if region.contains(p):pts.append(p)
    return pts
def labelled_geometries(unit):
    area=unit['region']
    if unit['kind']=='plot':
        idx=ctree.query(unit['full_geometry'],predicate='intersects')
        pos=[];neg=[]
        for j in idx:
            g=cgeoms[int(j)].intersection(area)
            g=polygon_only(g)
            if g is None or g.is_empty or g.area==0:continue
            if int(CROWNS.iloc[int(j)].Class_ID)==1:pos.append(g)
            else:neg.append(g)
        return unary_union(pos) if pos else None,unary_union(neg) if neg else None
    idx=cctree.query(unit['full_geometry'],predicate='intersects')
    pos=[polygon_only(ccgeoms[int(j)].intersection(area)) for j in idx]
    pos=[g for g in pos if g is not None and not g.is_empty and g.area>0]
    p=unary_union(pos) if pos else None
    # Exclude a 0.5 m boundary buffer around positive crowns to reduce label ambiguity.
    neg=polygon_only(area.difference(p.buffer(.5))) if p is not None else area
    return p,neg
POINTS=[]
for r in units.to_dict('records'):
    p,n=labelled_geometries(r)
    seed=int(hashlib.sha256(f"{r['group']}|{r['unit_key']}".encode()).hexdigest()[:8],16)
    for label,geo in [(1,p),(0,n)]:
        if geo is None or geo.is_empty or geo.area<.25:continue
        for pt in point_candidates(geo,SAMPLE_PER_CLASS_PER_UNIT,seed+label):
            POINTS.append(dict(group=r['group'],site=r['site'],kind=r['kind'],
                unit_key=r['unit_key'],fold=r['fold'],
                x=float(pt.x),y=float(pt.y),label=label))
candidates=pd.DataFrame(POINTS)
assert not candidates.empty
candidates.to_csv(NEW_OUT/'04_all_unit_candidate_points.csv',index=False)
print('Candidate points from original exhaustive annotations:')
display(candidates.groupby(['group','kind','fold','label']).agg(
    units=('unit_key','nunique'),points=('label','size')).reset_index())


In [ ]:
# 07A — Bounded native reads for RGB and each spectral band
# Unlike V4.007, use band 1 regardless of alpha for every survey; record alpha
# separately if present. A zero signal is invalid until source diagnosis.
# Every point is read via <= 8x8 native windows; no global raster resampling.
from rasterio.warp import transform as xy_transform
from rasterio.windows import Window
PATCH=8
def native_median(ds,x,y,band=1):
    rr,cc=ds.index(x,y)
    if rr<4 or cc<4 or rr>=ds.height-4 or cc>=ds.width-4:
        return np.nan,0.
    win=Window(cc-4,rr-4,PATCH,PATCH)
    z=ds.read(band,window=win,masked=True)
    a=np.asarray(z.data,np.float32)
    good=(~np.ma.getmaskarray(z))&np.isfinite(a)
    if ds.nodata is not None:good&=a!=ds.nodata
    good&=a!=0
    return (float(np.median(a[good])) if good.mean()>=.75 else np.nan,
            float(good.mean()))
OBS=[]
for group in GROUPS:
    g=candidates[candidates.group.eq(group)]
    if g.empty:continue
    print('MATCHING ALL ANNOTATION UNITS',group,'points',len(g),flush=True)
    rgbpath=DATA/'Ortho_RGB'/MOSAIC_KEY[group]
    # Avoid reading 3 RGB channels for each sample separately.
    with rasterio.Env(GDAL_CACHEMAX=12):
        from contextlib import ExitStack
        with ExitStack() as stack:
            rgb=stack.enter_context(rasterio.open(rgbpath))
            opened={str(p):stack.enter_context(rasterio.open(p)) for p in set(BANDS[group].values())}
            for j,r in enumerate(g.itertuples(index=False)):
                rr,cc=rgb.index(float(r.x),float(r.y))
                if rr<4 or cc<4 or rr>=rgb.height-4 or cc>=rgb.width-4:continue
                arr=rgb.read([1,2,3],window=Window(cc-4,rr-4,PATCH,PATCH),masked=True)
                rgbv=np.asarray(arr.data,np.float32)
                rgbmask=(~np.ma.getmaskarray(arr).any(axis=0))
                if rgbmask.mean()<.75:continue
                rv=np.median(rgbv[:,rgbmask],axis=1)/255.
                record=dict(group=group,site=r.site,kind=r.kind,
                    unit_key=r.unit_key,fold=int(r.fold),label=int(r.label),
                    x=float(r.x),y=float(r.y),R=rv[0],G=rv[1],B=rv[2])
                for band in ('red','green','red_edge','nir'):
                    ds=opened[str(BANDS[group][band])]
                    xx,yy=float(r.x),float(r.y)
                    if ds.crs!=rgb.crs:
                        cx,cy=xy_transform(rgb.crs,ds.crs,[xx],[yy]);xx,yy=cx[0],cy[0]
                    bidx={'red':1,'green':2,'red_edge':3,'nir':4}[band] if group=='Baego_MS' else 1
                    v,f=native_median(ds,xx,yy,bidx)
                    record[f'MS_{band}']=v
                    record[f'valid_{band}']=f
                OBS.append(record)
                if (j+1)%250==0:print(' ',group,j+1,'/',len(g),flush=True)
    gc.collect()
observed=pd.DataFrame(OBS)
assert not observed.empty
observed['valid3']=observed[['MS_green','MS_red_edge','MS_nir']].notna().all(axis=1)
observed['valid4']=observed['valid3']&observed.MS_red.notna()
observed.to_csv(NEW_OUT/'04_matched_original_units_ms.csv',index=False)
print('Matched units with valid 3/4 band observations:')
display(observed.groupby(['group','kind','fold']).agg(
    units=('unit_key','nunique'),points=('label','size'),
    good3=('valid3','sum'),good4=('valid4','sum')).reset_index())
print('MOS 5–6 band-1 zero red: will be excluded from 4-band comparison unless valid measured.')


In [ ]:
# 07B — Fold-level matched-support AUC with original unit CV, guard exclusion
# Compare RGB / RGB+3MS / RGB+4MS on IDENTICAL pixels for each pair.
# Non-independent point count is shown; independent unit counts govern acceptance.
def score_experiment(data,channels,fold,group):
    q=data[(data.group==group)&data[channels].notna().all(axis=1)]
    test=q[q.fold.eq(fold)].copy()
    train=q[q.fold.ne(fold)].copy()
    # 15 m exclusion between *entire* held-out and training unit polygons.
    valunits=units[(units.group==group)&units.fold.eq(fold)]
    trunits=units[(units.group==group)&~units.fold.eq(fold)]
    if len(valunits):
        exclusion=unary_union([g.buffer(GUARD_M) for g in valunits.full_geometry])
        unsafe=set(trunits[trunits.full_geometry.apply(lambda g:g.intersects(exclusion))].unit_key)
        train=train.loc[~train.unit_key.isin(unsafe)]
    if train.unit_key.nunique()<4 or test.unit_key.nunique()<2 or len(train)<12 or len(test)<6 or train.label.nunique()<2 or test.label.nunique()<2:
        return dict(group=group,fold=fold,features='+'.join(channels),status='insufficient spatially independent units',
            train_units=train.unit_key.nunique(),val_units=test.unit_key.nunique(),n_train=len(train),n_val=len(test),auc=np.nan)
    assert set(train.unit_key).isdisjoint(set(test.unit_key))
    model=HistGradientBoostingClassifier(max_iter=60,max_depth=3,max_leaf_nodes=7,
        learning_rate=.06,l2_regularization=4,early_stopping=False,random_state=20261117)
    model.fit(train[channels],train.label)
    proba=model.predict_proba(test[channels])[:,1]
    return dict(group=group,fold=fold,features='+'.join(channels),status='evaluated',
        train_units=train.unit_key.nunique(),val_units=test.unit_key.nunique(),
        n_train=len(train),n_val=len(test),auc=roc_auc_score(test.label,proba))
rows=[]
RGB_C=['R','G','B']
THREE=['MS_green','MS_red_edge','MS_nir']
FOUR=['MS_red']+THREE
for group in GROUPS:
    for fold in (0,1,2):
        for extra in (THREE,FOUR):
            matched=observed[observed.group.eq(group)&observed[extra].notna().all(axis=1)]
            base=score_experiment(matched,RGB_C,fold,group)
            fusion=score_experiment(matched,RGB_C+extra,fold,group)
            rows.append(dict(group=group,fold=fold,comparison='RGB+3MS' if len(extra)==3 else 'RGB+4MS',
                n_train=base['n_train'],n_val=base['n_val'],
                train_units=base['train_units'],val_units=base['val_units'],
                rgb_auc=base['auc'],fusion_auc=fusion['auc'],
                delta_auc=fusion['auc']-base['auc'] if np.isfinite(base['auc']) and np.isfinite(fusion['auc']) else np.nan,
                status=base['status'] if base['status']!='evaluated' else fusion['status']))
RESULT=pd.DataFrame(rows)
RESULT.to_csv(NEW_OUT/'04_original_units_3fold_rgb_ms_screen.csv',index=False)
display(RESULT.round(3))
print('Evaluated paired differences by acquisition:')
display(RESULT[RESULT.status.eq('evaluated')].groupby(['group','comparison']).agg(
    folds=('fold','size'),mean_delta=('delta_auc','mean'),
    mean_rgb=('rgb_auc','mean'),mean_fusion=('fusion_auc','mean')).round(3))
print('Outputs:',NEW_OUT)


### Interpretation notes

- Models compare **small RGB pixel statistics**, not frozen U-Net inference.
- Band 1 in each source is treated as spectral signal. Zero signal is rejected. If MOS 5–6 red is actually nonzero in other areas, the 4-band analysis uses only those matched samples; otherwise, its 3-band analysis proceeds.
- Three-way repeated fold AUC is unit-separated by original plots/circles, subject to 15 m training-guard and removal of circles intersecting plots. Invalid/bbox-only pixels never count as usable training data.
- Optical registration, image acquisition dates and radiometric calibration need subsequent validation. The current numbers are exploratory, not definitive proof of canopy-cover improvement.
